In [1]:
import pandas as pd
import numpy as np

In [7]:
path = r'/Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data/ords_prods_merge_updated.pkl'

In [10]:
ords_prods_merge_updated = pd.read_pickle(path)

In [11]:
ords_prods_merge_updated.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,prices,price_range_loc,day_type,busiest_days_new,busiest_period_of_day
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7,9.0,Mid-range product,Regularly busy,Regularly busy,Average orders
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16,12.5,Mid-range product,Regularly busy,Regularly busy,Average orders
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19,4.4,Low-range product,Regularly busy,Regularly busy,Average orders
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19,4.7,Low-range product,Regularly busy,Regularly busy,Average orders
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17,1.0,Low-range product,Regularly busy,Regularly busy,Average orders


In [12]:
ords_prods_merge_updated.groupby('department_id')['order_number'].mean()

department_id
1     15.457687
2     17.277920
3     17.179756
4     17.811403
5     15.215751
6     16.439806
7     17.225773
8     15.340520
9     15.895474
10    20.197148
11    16.170371
12    15.887622
13    16.583304
14    16.759763
15    16.165037
16    17.663925
17    15.694469
18    19.310514
19    17.177343
20    16.473447
21    22.902379
Name: order_number, dtype: float64

# Analysis
When comparing the results for the subset of one million rows with the results from the entire dataframe, we can observe small differences in the mean order numbers per department.
The averages for the full dataframe are slightly more stable and accurate, since they take into account all available customer orders. In contrast, the subset may have included fewer transactions for some departments, leading to minor variations in the averages.
Despite these differences, the overall pattern remains consistent, departments with higher mean order numbers in the subset continue to show higher values in the full dataset. This confirms that the subset provided a representative view of the data, but the full dataframe gives a more reliable and complete picture of customer behavior across all departments.

# Creating a loyalty flag for existing customers using the transform() and loc() functions

In [13]:
import pandas as pd
import numpy as np

In [15]:
# max_order column

ords_prods_merge_updated['max_order'] = ords_prods_merge_updated.groupby(['user_id'])['order_number'].transform('max')

In [16]:
# Loyal customers: more than 40 orders
ords_prods_merge_updated.loc[ords_prods_merge_updated['max_order'] > 40, 'loyalty_flag'] = 'Loyal customer'

# Regular customers: more than 10 but up to 40 orders
ords_prods_merge_updated.loc[(ords_prods_merge['max_order'] <= 40) & (ords_prods_merge_updated['max_order'] > 10), 'loyalty_flag'] = 'Regular customer'

# New customers: 10 orders or fewer
ords_prods_merge_updated.loc[ords_prods_merge_updated['max_order'] <= 10, 'loyalty_flag'] = 'New customer'

In [17]:
# Confirmation
ords_prods_merge_updated[['user_id', 'order_number', 'max_order', 'loyalty_flag']].head(15)

,user_id,order_number,max_order,loyalty_flag
0,1,1,10,New customer
1,1,1,10,New customer
2,1,1,10,New customer
3,1,1,10,New customer
4,1,1,10,New customer
5,1,2,10,New customer
6,1,2,10,New customer
7,1,2,10,New customer
8,1,2,10,New customer
9,1,2,10,New customer


In [18]:
ords_prods_merge_updated['loyalty_flag'].value_counts(dropna=False)

loyalty_flag
Regular customer    15890123
Loyal customer      10293366
New customer         6248971
Name: count, dtype: int64

# Analysis 
The results show that the majority of Instacart customers fall into the “Regular customer” category (around 15.9 million users). This indicates that most customers have placed between 10 and 40 orders, suggesting a stable base of moderately engaged users.
The “Loyal customer” group (about 10.2 million users) represents highly active customers who have placed more than 40 orders, this is a valuable segment for retention and reward programs.
Finally, there are approximately 6.2 million “New customers”, meaning users who have placed 10 or fewer orders. These customers could be targeted with onboarding or promotional campaigns to increase their engagement and move them toward the regular or loyal categories.

# The marketing team at Instacart wants to know whether there’s a difference between the spending habits of the three types of customers you identified

In [19]:
ords_prods_merge_updated.groupby('loyalty_flag')['prices'].describe()

,count,mean,std,min,25%,50%,75%,max
loyalty_flag,,,,,,,,
Loyal customer,10293366.0,10.388902,327.870016,1.0,4.2,7.4,11.2,99999.0
New customer,6248971.0,13.294907,597.322095,1.0,4.2,7.4,11.3,99999.0
Regular customer,15890123.0,12.496618,539.494200,1.0,4.2,7.4,11.3,99999.0


# Comment 
There’s no major difference in average spending between the three customer groups, all have a median price around $7.4. However, new and regular customers show higher standard deviations, meaning their spending is more variable compared to loyal customers.

# Target different types of spenders in their marketing campaigns

In [22]:
import numpy as np

ords_prods_merge_updated['prices'] = pd.to_numeric(ords_prods_merge_updated['prices'], errors='coerce')

In [23]:
# 2) Average price per user 
ords_prods_merge_updated['spending_mean'] = (
    ords_prods_merge_updated.groupby('user_id')['prices'].transform('mean'))

In [24]:
# 3) Flag based on the rule: < 10 = Low, >= 10 = High
ords_prods_merge_updated['spending_flag'] = np.where(
    ords_prods_merge_updated['spending_mean'] < 10,
    'Low spender',
    'High spender')

In [25]:
# Quick check
ords_prods_merge_updated['spending_flag'].value_counts()

spending_flag
Low spender     31797024
High spender      635436
Name: count, dtype: int64

# Comments
Most customers fall into the Low spender category, with an average purchase price below $10. Only a small portion are High spenders, meaning marketing campaigns could focus on encouraging higher-value purchases among low spenders or rewarding loyalty among the smaller high-spender group.

# Determine frequent versus non-frequent customers

In [26]:
# 1) Calculate the median of 'days_since_prior_order' per user
ords_prods_merge_updated['median_days'] = (
    ords_prods_merge_updated.groupby('user_id')['days_since_prior_order'].transform('median'))

In [27]:
# 2) Create the frequency flag based on the given criteria
ords_prods_merge_updated.loc[ords_prods_merge_updated['median_days'] > 20, 'frequency_flag'] = 'Non-frequent customer'
ords_prods_merge_updated.loc[
    (ords_prods_merge_updated['median_days'] > 10) & (ords_prods_merge_updated['median_days'] <= 20),
    'frequency_flag'
] = 'Regular customer'
ords_prods_merge_updated.loc[ords_prods_merge_updated['median_days'] <= 10, 'frequency_flag'] = 'Frequent customer'

In [28]:
# 3) Quick check
ords_prods_merge_updated['frequency_flag'].value_counts()

frequency_flag
Frequent customer        21576343
Regular customer          7216768
Non-frequent customer     3639349
Name: count, dtype: int64

# Comments
Most customers fall into the Frequent customer group, meaning they place orders quite often (every 10 days or less). A smaller portion are Regular customers (10–20 days between orders), and the rest are Non-frequent customers who order less regularly. This segmentation can help the marketing team tailor reminders or reorder prompts based on how often each group shops.

# Export

In [30]:
import os


In [33]:
print(path)

/Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data/ords_prods_merge_updated.pkl


In [34]:
path = r'/Users/mariatirado/29-10-2025 Instacart Basket Analysis'

In [38]:
import os

ords_prods_merge_updated.to_pickle(os.path.join(
    path, '02 Data', 'Prepared Data', 'ords_prods_merge_spending_frequency.pkl'))

In [39]:
os.path.exists(os.path.join(path, '02 Data', 'Prepared Data', 'ords_prods_merge_spending_frequency.pkl'))

True